# Ray Tune: Distributed Hyperparameter Optimization

## What Is Ray Tune?

Imagine tuning a car engine, but instead of one mechanic trying one setting at a time,  
you have 50 mechanics working in parallel — each trying a different combination.  
A supervisor (scheduler) watches the results and stops the clearly bad cars early, redirecting those mechanics to better combinations.

**Ray Tune** is that system — it runs hyperparameter trials in **parallel across a cluster**,  
with intelligent schedulers that stop bad trials early (like Optuna pruning, but distributed).

Key features:
- **Distributed**: scale from 1 laptop to 1000 cloud machines with no code changes
- **Early stopping**: ASHA, HyperBand, Population-Based Training (PBT)
- **Integrations**: works with PyTorch, TensorFlow, sklearn, XGBoost, Optuna, Hyperopt
- **Search algorithms**: random, Bayesian (Optuna/Hyperopt backend), CMA-ES, BOHB
- **Fault tolerant**: automatically retries failed trials
- **Checkpointing**: saves trial state so long runs can be resumed

## Resources

- **Docs**: [https://docs.ray.io/en/latest/tune/](https://docs.ray.io/en/latest/tune/)
- **GitHub**: [https://github.com/ray-project/ray](https://github.com/ray-project/ray)
- **YouTube**: [https://www.youtube.com/watch?v=M_d3FUx69sA](https://www.youtube.com/watch?v=M_d3FUx69sA)
- **Paper**: [Tune: A Research Platform for Distributed Model Selection](https://arxiv.org/abs/1807.05118)

## Installation

```bash
pip install 'ray[tune]'
# With optional search algorithm backends:
pip install 'ray[tune]' optuna hyperopt
```

In [ ]:
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

try:
    import ray
    from ray import tune
    from ray.tune import CLIReporter
    from ray.tune.schedulers import ASHAScheduler, HyperBandScheduler
    RAY_AVAILABLE = True
    print(f"Ray version: {ray.__version__}")
except ImportError:
    RAY_AVAILABLE = False
    print("Ray not installed — simulated output shown.")
    print("Install: pip install 'ray[tune]'")

from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

X, y = make_classification(n_samples=2000, n_features=20, n_informative=12, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Dataset: {X_train.shape[0]} train, {X_test.shape[0]} test, {X.shape[1]} features")

## Core Concept 1: The Trainable Function

Ray Tune's basic API: define a **trainable function** that takes a `config` dict,  
trains a model, and reports metrics using `tune.report()`.

In [ ]:
if RAY_AVAILABLE:
    # Initialize Ray
    if not ray.is_initialized():
        ray.init(ignore_reinit_error=True, num_cpus=4)

    def train_rf(config):
        """Trainable function: receives config, reports metrics."""
        model = RandomForestClassifier(
            n_estimators=config['n_estimators'],
            max_depth=config['max_depth'],
            min_samples_split=config['min_samples_split'],
            random_state=42, n_jobs=1,  # use n_jobs=1 — Ray parallelizes at trial level
        )
        # 3-fold CV
        scores = cross_val_score(model, X_train, y_train, cv=3, scoring='roc_auc')

        # Report metric to Ray Tune
        tune.report(mean_auc=scores.mean(), std_auc=scores.std())

    # Search space
    config = {
        'n_estimators':    tune.randint(50, 500),
        'max_depth':       tune.randint(2, 15),
        'min_samples_split': tune.randint(2, 20),
    }

    # Run tuning
    t0 = time.time()
    analysis = tune.run(
        train_rf,
        config=config,
        num_samples=20,          # number of trials
        metric='mean_auc',
        mode='max',
        verbose=0,
        name='rf_tune',
    )

    best_config = analysis.get_best_config(metric='mean_auc', mode='max')
    best_result = analysis.best_result
    print(f"Tuning complete in {time.time()-t0:.1f}s")
    print(f"Best AUC:    {best_result['mean_auc']:.4f}")
    print(f"Best config: {best_config}")

    ray.shutdown()

else:
    print("Ray Tune basic usage (simulated):")
    print()
    print("  def train_rf(config):")
    print("      model = RandomForestClassifier(")
    print("          n_estimators=config['n_estimators'],")
    print("          max_depth=config['max_depth'],")
    print("      )")
    print("      scores = cross_val_score(model, X_train, y_train, cv=3, scoring='roc_auc')")
    print("      tune.report(mean_auc=scores.mean())   # ← key: report metric")
    print()
    print("  analysis = tune.run(")
    print("      train_rf,")
    print("      config={'n_estimators': tune.randint(50, 500), 'max_depth': tune.randint(2, 15)},")
    print("      num_samples=20,")
    print("      metric='mean_auc',")
    print("      mode='max',")
    print("  )")
    print()
    print("  Best AUC:    0.9154")
    print("  Best config: {n_estimators: 287, max_depth: 9, min_samples_split: 4}")

## Core Concept 2: Search Space — tune.* Distributions

Ray Tune has its own search space definition API — similar to Optuna but in dict form.

In [ ]:
print("Ray Tune search space functions:")
print()

search_space_examples = [
    ('tune.randint(low, high)',       'Integer in [low, high)',          'n_estimators: tune.randint(10, 500)'),
    ('tune.uniform(low, high)',       'Float in [low, high] uniform',    'subsample: tune.uniform(0.5, 1.0)'),
    ('tune.loguniform(low, high)',    'Float in log scale',              'lr: tune.loguniform(1e-5, 1e-1)'),
    ('tune.choice([a, b, c])',        'One of the listed values',        'optimizer: tune.choice(["adam", "sgd"])'),
    ('tune.grid_search([v1, v2])',    'Try all listed values (grid)',     'batch_size: tune.grid_search([32, 64, 128])'),
    ('tune.sample_from(fn)',          'Custom lambda function',           'n_heads: tune.sample_from(lambda _: 2**np.random.randint(1,4))'),
    ('tune.quniform(low, high, q)',   'Quantized uniform (multiples of q)','lr_steps: tune.quniform(100, 1000, 100)'),
]

for func, desc, example in search_space_examples:
    print(f"  {func}")
    print(f"    Description: {desc}")
    print(f"    Example:     {example}")
    print()

print("Full config example:")
print("  config = {")
print("      'lr':           tune.loguniform(1e-5, 1e-1),")
print("      'batch_size':   tune.choice([16, 32, 64, 128]),")
print("      'n_layers':     tune.randint(1, 5),")
print("      'hidden_size':  tune.randint(32, 512),")
print("      'dropout':      tune.uniform(0.0, 0.5),")
print("      'optimizer':    tune.choice(['Adam', 'AdamW', 'SGD']),")
print("  }")

## Core Concept 3: Schedulers — Early Stopping for Trials

Schedulers decide which trials to stop early and which to keep running.  
This is Ray Tune's key advantage: instead of running every trial to completion, it kills the bad ones early.

In [ ]:
print("Ray Tune schedulers:")
print()

schedulers = [
    {
        'name': 'ASHAScheduler (Asynchronous Successive Halving)',
        'code': """ASHAScheduler(
    metric='val_loss',
    mode='min',
    max_t=100,       # max epochs
    grace_period=10, # run at least 10 epochs before considering pruning
    reduction_factor=3  # keep top 1/3 of trials at each rung
)""",
        'how':  """Runs trials in 'rungs'. After grace_period epochs, top 1/reduction_factor trials
    advance. Others are stopped. Very efficient for many short trials.""",
        'best_for': 'Neural network training where epochs are the 'step'",
    },
    {
        'name': 'HyperBandScheduler',
        'code': "HyperBandScheduler(metric='val_acc', mode='max', max_t=81)",
        'how':  """Synchronized version of successive halving. Runs brackets of trials,
    progressively eliminating the worst performers.""",
        'best_for': 'When you want theoretically optimal sample complexity',
    },
    {
        'name': 'PopulationBasedTraining (PBT)',
        'code': """PopulationBasedTraining(
    metric='val_acc', mode='max',
    perturbation_interval=10,  # exploit/explore every 10 epochs
    hyperparam_mutations={'lr': tune.loguniform(1e-4, 1e-1)},
)""",
        'how':  """Evolution-inspired: worst trials 'copy' hyperparams from best trials
    and add noise. Hyperparameters can change during training.""",
        'best_for': 'Very long NN training; explores hyperparameter schedule (not just fixed params)',
    },
]

for s in schedulers:
    print(f"  {s['name']}:")
    print(f"    Code:")
    for line in s['code'].strip().split('\n'):
        print(f"      {line}")
    print(f"    How it works: {s['how'].strip()}")
    print(f"    Best for: {s['best_for']}")
    print()

print("Rule: Start with ASHAScheduler — it's the most practical and widely used.")

## Core Concept 4: Ray Tune + ASHA for Neural Network HPO

In [ ]:
NN_TUNE_CODE = '''
import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler
import torch
import torch.nn as nn

def build_model(config):
    layers = []
    in_size = INPUT_DIM
    for _ in range(config['n_layers']):
        layers += [
            nn.Linear(in_size, config['hidden_size']),
            nn.ReLU(),
            nn.Dropout(config['dropout']),
        ]
        in_size = config['hidden_size']
    layers.append(nn.Linear(in_size, NUM_CLASSES))
    return nn.Sequential(*layers)

def train_nn(config, checkpoint_dir=None):
    model = build_model(config)
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
    criterion = nn.CrossEntropyLoss()

    # Load checkpoint if resuming
    start_epoch = 0
    if checkpoint_dir:
        state = torch.load(os.path.join(checkpoint_dir, 'checkpoint.pt'))
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        start_epoch = state['epoch'] + 1

    for epoch in range(start_epoch, config['max_epochs']):
        # Training loop
        model.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()

        # Validation
        val_loss, val_acc = evaluate(model, val_loader)

        # Save checkpoint
        with tune.checkpoint_dir(epoch) as checkpoint_dir:
            torch.save(
                {'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'epoch': epoch},
                os.path.join(checkpoint_dir, 'checkpoint.pt')
            )

        # Report to Ray Tune (ASHAScheduler uses this to prune)
        tune.report(val_loss=val_loss, val_acc=val_acc)

# Search space
config = {
    'lr':          tune.loguniform(1e-5, 1e-1),
    'hidden_size': tune.choice([64, 128, 256, 512]),
    'n_layers':    tune.randint(1, 5),
    'dropout':     tune.uniform(0.0, 0.5),
    'max_epochs':  100,
}

scheduler = ASHAScheduler(
    metric='val_loss',
    mode='min',
    max_t=100,          # max epochs
    grace_period=10,    # min epochs before considering pruning
    reduction_factor=3, # top 1/3 survive each rung
)

result = tune.run(
    train_nn,
    config=config,
    num_samples=50,     # start 50 trials
    scheduler=scheduler,
    # Resources per trial:
    resources_per_trial={'cpu': 2, 'gpu': 0.5},  # each trial gets 2 CPU + 0.5 GPU
    local_dir='~/ray_results',
)

best_trial = result.get_best_trial('val_loss', 'min', 'last')
print(f"Best val_loss: {best_trial.last_result['val_loss']:.4f}")
print(f"Best config:   {best_trial.config}")
'''

print("Ray Tune + ASHA for neural network HPO:")
print(NN_TUNE_CODE)

print("How ASHA prunes:")
print("  50 trials start, all run to epoch 10 (grace_period)")
print("  At epoch 10: keep top 50/3 = 17 trials, stop 33")
print("  At epoch 30: keep top 17/3 = 6 trials, stop 11")
print("  At epoch 90: keep top 6/3 = 2 trials, stop 4")
print("  Result: only 2 trials run to 100 epochs — massive compute savings!")

## Core Concept 5: Search Algorithms — Bayesian via Optuna Backend

In [ ]:
if RAY_AVAILABLE:
    try:
        from ray.tune.search.optuna import OptunaSearch
        OPTUNA_SEARCH = True
    except ImportError:
        OPTUNA_SEARCH = False

    if OPTUNA_SEARCH:
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)

        if not ray.is_initialized():
            ray.init(ignore_reinit_error=True, num_cpus=4)

        def train_fn(config):
            model = GradientBoostingClassifier(
                n_estimators=config['n_estimators'],
                max_depth=config['max_depth'],
                learning_rate=config['lr'],
                random_state=42,
            )
            scores = cross_val_score(model, X_train, y_train, cv=3, scoring='roc_auc')
            tune.report(mean_auc=scores.mean())

        search_alg = OptunaSearch(metric='mean_auc', mode='max')

        analysis = tune.run(
            train_fn,
            config={
                'n_estimators': tune.randint(50, 300),
                'max_depth':    tune.randint(2, 10),
                'lr':           tune.loguniform(0.01, 0.3),
            },
            num_samples=20,
            search_alg=search_alg,
            verbose=0,
        )

        print(f"Optuna search via Ray Tune:")
        print(f"  Best AUC: {analysis.best_result['mean_auc']:.4f}")
        print(f"  Best config: {analysis.best_config}")
        ray.shutdown()

else:
    print("Bayesian search via OptunaSearch backend (simulated):")
    print()
    print("  from ray.tune.search.optuna import OptunaSearch")
    print()
    print("  search_alg = OptunaSearch(metric='mean_auc', mode='max')")
    print()
    print("  analysis = tune.run(")
    print("      train_fn,")
    print("      config={...},")
    print("      num_samples=20,")
    print("      search_alg=search_alg,   # ← Optuna does the sampling")
    print("      scheduler=ASHAScheduler(),  # ← ASHA does the pruning")
    print("  )")
    print()
    print("  Combined: Optuna proposes SMART configs, ASHA kills bad trials early")
    print("  → Best of both worlds!")
    print()
    print("  Other search algorithm backends:")
    print("    HyperOptSearch  — Tree Parzen Estimator (older Bayesian)")
    print("    BayesOptSearch  — Gaussian Process-based")
    print("    CMAEvolutionarySearch  — Evolution strategy")
    print("    BasicVariantGenerator — Random search (default)")

## Core Concept 6: Distributed Scaling

Ray Tune scales from a single laptop to 1000 cloud machines with **no code changes**.
The `resources_per_trial` tells Ray how to allocate resources across your cluster.

In [ ]:
print("Ray Tune distributed scaling:")
print()

scaling_examples = [
    {
        'scenario': 'Local laptop (8 CPU, 1 GPU)',
        'code': """ray.init(num_cpus=8, num_gpus=1)
tune.run(
    train_fn,
    resources_per_trial={'cpu': 2, 'gpu': 0.25},  # 4 trials in parallel
    num_samples=40,
)""",
        'parallel': '4 trials run simultaneously (8 CPU / 2 = 4)',
    },
    {
        'scenario': 'AWS cluster (10 × 8-core + 2 GPU machines)',
        'code': """# Connect to existing Ray cluster
ray.init(address='auto')   # connects to cluster started with ray start --head
tune.run(
    train_fn,
    resources_per_trial={'cpu': 4, 'gpu': 1},  # each trial gets 4 CPU + 1 GPU
    num_samples=100,
)""",
        'parallel': '20 trials run simultaneously (10 machines × 2 GPU each)',
    },
    {
        'scenario': 'Kubernetes (auto-scaled)',
        'code': """# KubeRay — Ray on Kubernetes
# kubectl apply -f ray_cluster.yaml
# Ray auto-scales workers based on pending trials
ray.init(address='ray://my-ray-cluster:10001')
tune.run(train_fn, resources_per_trial={'cpu': 2, 'gpu': 1}, num_samples=500)""",
        'parallel': 'Auto-scales to handle pending trials; scales down when done',
    },
]

for ex in scaling_examples:
    print(f"  Scenario: {ex['scenario']}")
    print(f"  Code:")
    for line in ex['code'].strip().split('\n'):
        print(f"    {line}")
    print(f"  Parallelism: {ex['parallel']}")
    print()

print("Key insight: same training code works locally and on a cluster.")
print("Only ray.init() changes — pointing to local or cluster.")

## Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Using `n_jobs=-1` inside trial | Over-subscribes CPU, OOM | Use `n_jobs=1` — Ray parallelizes at trial level |
| Not calling `tune.report()` | Trial hangs | Always call `tune.report(metric=value)` every step |
| Sending large data to each trial | Slow startup | Use `ray.put(data)` and pass ObjectRef |
| ASHA without `grace_period` | Good trials killed too early | Set `grace_period` to at least 10% of max_t |
| Forgetting `ray.init()` | Runs on single core | Always `ray.init()` before `tune.run()` |
| GPU not detected | `RuntimeError: No GPU` | Set `CUDA_VISIBLE_DEVICES` and check `ray.available_resources()` |

## Mini Project: XGBoost HPO with Ray Tune + ASHA

In [ ]:
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

if RAY_AVAILABLE and XGB_AVAILABLE:
    if not ray.is_initialized():
        ray.init(ignore_reinit_error=True, num_cpus=4)

    def train_xgb(config):
        model = xgb.XGBClassifier(
            n_estimators=config['n_estimators'],
            max_depth=config['max_depth'],
            learning_rate=config['lr'],
            subsample=config['subsample'],
            colsample_bytree=config['colsample'],
            reg_alpha=config['reg_alpha'],
            use_label_encoder=False,
            eval_metric='logloss',
            random_state=42, n_jobs=1,
        )
        scores = cross_val_score(model, X_train, y_train, cv=3, scoring='roc_auc')
        tune.report(mean_auc=scores.mean())

    config = {
        'n_estimators': tune.randint(100, 500),
        'max_depth':    tune.randint(3, 10),
        'lr':           tune.loguniform(0.01, 0.3),
        'subsample':    tune.uniform(0.5, 1.0),
        'colsample':    tune.uniform(0.5, 1.0),
        'reg_alpha':    tune.loguniform(1e-8, 10.0),
    }

    t0 = time.time()
    analysis = tune.run(
        train_xgb,
        config=config,
        num_samples=30,
        metric='mean_auc',
        mode='max',
        verbose=0,
    )

    best = analysis.get_best_trial('mean_auc', 'max', 'last')
    print(f"=" * 60)
    print(f"XGBOOST HPO WITH RAY TUNE")
    print(f"=" * 60)
    print(f"Trials: 30 | Time: {time.time()-t0:.1f}s")
    print(f"Best AUC:    {best.last_result['mean_auc']:.4f}")
    print(f"Best config: {best.config}")

    # Final model with best config
    best_model = xgb.XGBClassifier(**{k: v for k, v in best.config.items()},
                                    use_label_encoder=False, eval_metric='logloss',
                                    random_state=42)
    best_model.fit(X_train, y_train)
    test_auc = roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])
    print(f"Test AUC:    {test_auc:.4f}")
    ray.shutdown()

else:
    print("XGBoost HPO with Ray Tune (simulated):")
    print()
    print("  30 parallel trials, 6 hyperparameters")
    print()
    print("  Best AUC:    0.9287")
    print("  Best config:")
    print("    n_estimators: 387")
    print("    max_depth: 6")
    print("    lr: 0.0312")
    print("    subsample: 0.812")
    print("    colsample: 0.743")
    print("    reg_alpha: 0.0021")
    print()
    print("  Test AUC: 0.9271")
    print("  Default XGB AUC: 0.9134  (+1.5% improvement)")

## Interview Questions and Answers

In [ ]:
qa = [
    {"q": "Ray Tune vs Optuna — which would you choose and when?",
     "a": """Both are excellent HPO frameworks. The key difference is scale.

Optuna:
  - Single machine, sequential or limited parallel trials
  - Simpler API: objective function returns a value
  - Pruning via trial.report() + trial.should_prune()
  - Great for: < 100 trials, local development, sklearn/LightGBM tuning
  - Integrates natively with MLflow for tracking

Ray Tune:
  - Single machine to 1000-node cluster, no code changes
  - Trainable function reports metrics via tune.report()
  - ASHA/HyperBand schedulers (more scalable than Optuna MedianPruner)
  - Can use Optuna as backend: OptunaSearch
  - Built-in checkpointing and fault tolerance for long runs
  - Great for: NN training, GPU clusters, > 100 trials, distributed

Rule:
  Local, quick HPO → Optuna (simpler, sufficient)
  GPU cluster, many trials, NN training → Ray Tune
  Best of both → Ray Tune with OptunaSearch + ASHAScheduler"""},

    {"q": "How does ASHA work and why is it efficient?",
     "a": """ASHA = Asynchronous Successive Halving Algorithm.

Successive Halving idea:
  1. Start N trials with budget B/N each
  2. Promote top 1/eta trials to next rung (more budget)
  3. Repeat until 1 trial gets full budget B
  → Costs ~same compute as running 1 trial fully but tests N configs!

Example (N=81, B=81 epochs, eta=3):
  Rung 1: 81 trials × 1 epoch  → cost: 81 GPU-epochs
  Rung 2: 27 trials × 3 epochs → cost: 81 GPU-epochs
  Rung 3: 9 trials  × 9 epochs → cost: 81 GPU-epochs
  Rung 4: 3 trials  × 27 epoch → cost: 81 GPU-epochs
  Rung 5: 1 trial   × 81 epoch → cost: 81 GPU-epochs
  Total: 405 GPU-epochs vs 81×81=6561 for full grid search
  → 16× more efficient!

ASHA = Asynchronous:
  Synchronous SHA waits for all trials at a rung.
  ASHA promotes immediately — no idle workers.
  Better for heterogeneous trial durations.

Assumption: early performance correlates with final performance.
This is true for NN training (loss at epoch 5 predicts final loss)."""},

    {"q": "What is Population-Based Training and when would you use it?",
     "a": """PBT (Population-Based Training) combines hyperparameter search
with learning schedule optimization.

Standard HPO: find fixed hyperparameters (lr=0.001 for all 100 epochs)
PBT: find hyperparameter SCHEDULES (lr starts at 0.01, drops to 0.001 at epoch 50)

How PBT works:
  1. Start N trials with random hyperparameters
  2. Every T steps (e.g., 10 epochs):
     a. Rank trials by current metric
     b. Bottom 20% EXPLOIT: copy weights + hyperparams from top 20%
     c. Add noise (EXPLORE): perturb the copied hyperparams slightly
  3. Repeat — hyperparams evolve throughout training

Result: each trial's hyperparams adapt over time.
The population collectively discovers the best schedule.

When to use PBT:
  - Training is very long (> 100 epochs)
  - You believe hyperparams should change during training
  - RL training (PBT originated in DeepMind's RL research)

When NOT to use:
  - Short training (< 20 epochs) — not enough time to adapt
  - Tabular models (GBM, RF) — no epochs concept"""},

    {"q": "How would you use Ray Tune for a large-scale GPU training job?",
     "a": """End-to-end large-scale GPU HPO workflow:

1. Set up Ray cluster:
   # On head node:
   ray start --head --num-cpus=8 --num-gpus=4
   # On worker nodes:
   ray start --address='head_ip:6379' --num-cpus=8 --num-gpus=4

2. Connect and run:
   ray.init(address='auto')

   analysis = tune.run(
       train_nn,                      # your training function
       config={...},
       num_samples=100,               # 100 total trials
       resources_per_trial={'cpu': 4, 'gpu': 1},  # 1 GPU per trial
       scheduler=ASHAScheduler(max_t=200, grace_period=20),
       search_alg=OptunaSearch(metric='val_acc', mode='max'),
       local_dir='s3://bucket/tune_results',   # save to S3
       resume=True,                   # resume if cluster restarts
       checkpoint_freq=10,            # checkpoint every 10 epochs
       max_failures=3,                # retry failed trials
   )

3. Monitor in real time:
   # Open Ray Dashboard: http://head_ip:8265
   # See: trial status, GPU utilization, metrics over time

4. Get best model:
   best_checkpoint = analysis.best_checkpoint
   model.load_state_dict(torch.load(best_checkpoint))"""},
]

for i, item in enumerate(qa, 1):
    print(f"Q{i}: {item['q']}")
    print(f"A:  {item['a'].strip()}")
    print("-" * 65)
    print()

## Summary

| Concept | Ray Tune API |
|---------|-------------|
| Initialize | `ray.init(num_cpus=8)` or `ray.init(address='auto')` |
| Report metric | `tune.report(val_loss=loss, val_acc=acc)` |
| Random int | `tune.randint(low, high)` |
| Log-uniform float | `tune.loguniform(1e-5, 1e-1)` |
| Categorical | `tune.choice(['adam', 'sgd'])` |
| Grid values | `tune.grid_search([32, 64, 128])` |
| ASHA scheduler | `ASHAScheduler(metric='val_loss', mode='min', max_t=100)` |
| Bayesian search | `OptunaSearch(metric='val_acc', mode='max')` |
| Run tuning | `tune.run(fn, config=..., num_samples=N, scheduler=..., search_alg=...)` |
| Best config | `analysis.get_best_config(metric, mode)` |
| Best trial | `analysis.get_best_trial(metric, mode, 'last')` |
| GPU allocation | `resources_per_trial={'cpu': 2, 'gpu': 0.5}` |
| Stop Ray | `ray.shutdown()` |

### Next Steps
1. **Ray Tune docs**: [https://docs.ray.io/en/latest/tune/getting-started.html](https://docs.ray.io/en/latest/tune/getting-started.html)
2. **Ray Tune examples**: [https://docs.ray.io/en/latest/tune/examples/](https://docs.ray.io/en/latest/tune/examples/)
3. **PBT paper**: [Population Based Training of Neural Networks](https://arxiv.org/abs/1711.09846)
4. **Next**: Capstone Projects — applying everything learned end-to-end